In [1]:
import os
import glob

# 1. Pfad zum Ordner definieren (Passe diesen Pfad an deinen Ordner an!)
# Wenn die Dateien im selben Ordner wie das Notebook liegen, kannst du '.' lassen.
folder_path = '../messurement'

# 2. Alle .txt-Dateien im Ordner finden
# (Nutzt glob, um nur .txt-Dateien zu matchen)
txt_files = glob.glob(os.path.join(folder_path, '*.txt'))

if not txt_files:
    print(f"Keine .txt-Dateien im Ordner '{folder_path}' gefunden. Bitte Pfad prüfen!")
else:
    print(f"Gefundene Dateien: {len(txt_files)}\n" + "-"*40)

    # 3. Jede Datei einzeln durchgehen
    for file_path in txt_files:
        total_nanoseconds = 0
        file_name = os.path.basename(file_path)
        
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                for line in file:
                    # Zeile säubern (Leerzeichen/Umbruche weg)
                    cleaned_line = line.strip()
                    
                    # Ignoriere leere Zeilen
                    if not cleaned_line:
                        continue
                        
                    # Extrahiere die Zahl (splittet bei " nanoseconds" oder Leerzeichen)
                    # Nimmt das erste Element (die Zahl)
                    number_part = cleaned_line.split()[0]
                    
                    # In Ganzzahl umwandeln und aufaddieren
                    total_nanoseconds += int(number_part)
            
            # Umrechnungen für bessere Lesbarkeit
            total_milliseconds = total_nanoseconds / 1_000_000
            total_seconds = total_nanoseconds / 1_000_000_000
            
            # Ergebnis ausgeben
            print(f"Datei: {file_name}")
            print(f"  -> Gesamtzeit: {total_nanoseconds:,} ns")
            print(f"  -> Entspricht: {total_milliseconds:.2f} ms ({total_seconds:.4f} Sekunden)")
            print("-" * 40)
            
        except Exception as e:
            print(f"Fehler beim Lesen von {file_name}: {e}")
            print("-" * 40)

Gefundene Dateien: 12
----------------------------------------
Datei: imu_results_rmw_fastrtps_dynamic_cpp.txt
  -> Gesamtzeit: 418,331,904 ns
  -> Entspricht: 418.33 ms (0.4183 Sekunden)
----------------------------------------
Datei: imu_results_rmw_fastrtps_cpp.txt
  -> Gesamtzeit: 421,799,862 ns
  -> Entspricht: 421.80 ms (0.4218 Sekunden)
----------------------------------------
Datei: camera_results_rmw_fastrtps_cpp.txt
  -> Gesamtzeit: 689,269,694 ns
  -> Entspricht: 689.27 ms (0.6893 Sekunden)
----------------------------------------
Datei: lidar_results_rmw_cyclonedds_cpp.txt
  -> Gesamtzeit: 425,097,189 ns
  -> Entspricht: 425.10 ms (0.4251 Sekunden)
----------------------------------------
Datei: imu_results_rmw_cyclonedds_cpp.txt
  -> Gesamtzeit: 504,705,189 ns
  -> Entspricht: 504.71 ms (0.5047 Sekunden)
----------------------------------------
Datei: camera_results_rmw_connextdds.txt
  -> Gesamtzeit: 799,042,176 ns
  -> Entspricht: 799.04 ms (0.7990 Sekunden)
------------

In [7]:
import os
import glob
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Pfad zum Ordner mit den Textdateien definieren
folder_path = '../messurement'  # Dein angepasster Pfad

# 2. Datenstrukturen vorbereiten
txt_files = glob.glob(os.path.join(folder_path, '*.txt'))
parsed_data = []

# 3. Dateien automatisiert auslesen und parsen
for file_path in txt_files:
    file_name = os.path.basename(file_path)
    
    # Regulärer Ausdruck für Sensor und RMW-Namen
    match = re.match(r"([a-zA-Z0-9]+)_results_(rmw_[a-zA-Z0-9_]+)\.txt", file_name)
    
    if not match:
        continue  
        
    sensor_type = match.group(1).capitalize()  # "Camera" oder "Lidar"
    rmw_name = match.group(2)                  # z.B. "rmw_fastrtps_cpp"
    
    total_nanoseconds = 0
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            entry_count = 0  # Zähler für die Einträge in der aktuellen Datei
            for line in file:
                cleaned_line = line.strip()
                if not cleaned_line:
                    continue
                
                # Zähler für gültige Einträge erhöhen
                entry_count += 1
                
                # Die ersten 50 Einträge verwerfen
                if entry_count <= 50:
                    continue
                
                number_part = cleaned_line.split()[0]
                total_nanoseconds += int(number_part)
                
        # In Millisekunden umrechnen
        total_milliseconds = total_nanoseconds / 1_000_000
        
        parsed_data.append({
            "Sensor": sensor_type,
            "RMW Implementation": rmw_name,
            "Gesamtzeit (ms)": total_milliseconds
        })
    except Exception as e:
        print(f"Fehler beim Parsen von {file_name}: {e}")

# 4. DataFrame erstellen und plotten
df = pd.DataFrame(parsed_data)

if df.empty:
    print("Keine passenden Daten zum Plotten gefunden. Überprüfe die Dateinamen!")
else:
    # Sortieren für eine saubere Reihenfolge der Balken
    df = df.sort_values(by="Gesamtzeit (ms)")

    # Plot-Design festlegen
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(11, 6))
    
    # Gruppiertes Balkendiagramm mit kontrastreichen, unterschiedlichen Farben ("Set2")
    barplot = sns.barplot(
        x="Sensor", 
        y="Gesamtzeit (ms)", 
        hue="RMW Implementation", 
        data=df, 
        palette="Set2"  
    )
    
    # Titel und Achsen beschriften
    plt.title("ROS 2 RMW Performance-Vergleich (Latenz - ohne die ersten 50 Werte)", fontsize=14, fontweight='bold', pad=15)
    plt.xlabel("Sensortyp", fontsize=12, labelpad=10)
    plt.ylabel("Gesamtzeit in ms", fontsize=12, labelpad=10)
    
    # Legende außerhalb des Diagramms platzieren
    plt.legend(title="RMW Middleware", bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
    
    # Exakte Werte über die Balken schreiben
    for p in barplot.patches:
        if p.get_height() > 0:
            barplot.annotate(
                f"{p.get_height():.1f} ms",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                xytext=(0, 8), 
                textcoords='offset points', 
                fontsize=9,
                fontweight='semibold'
            )
            
    plt.tight_layout()
    
    # Diagramm保存 und anzeigen
    plt.savefig("rmw_performance_vergleich.png", dpi=300)
    plt.show()  
    print("Das Diagramm wurde erfolgreich als 'rmw_performance_vergleich.png' gespeichert!")

Keine passenden Daten zum Plotten gefunden. Überprüfe die Dateinamen!
